# Data Integration
**Team:** Sage Kim & Kyna Tyagi

## 0. Load Libraries & Data

In [1]:
import pandas as pd
import numpy as np

# Load raw FAO data
fao = pd.read_csv('../data/raw/faostat_cereal_raw.csv')
wb  = pd.read_csv('../data/raw/worldbank_gdp_raw.csv', skiprows=4)

# OpenRefine operations (documented in docs/openrefine-history.json)

fao = fao[fao['Flag'] != 'M'].copy()

fao['Area'] = fao['Area'].str.strip()

fao = fao[['Area Code (M49)', 'Area', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description', 'Note']]

## 1. Explore Dataset

In [2]:
print(fao.shape)
print(fao.dtypes)
fao.head()

(2616, 8)
Area Code (M49)       int64
Area                 object
Year                  int64
Unit                 object
Value               float64
Flag                 object
Flag Description     object
Note                 object
dtype: object


,Area Code (M49),Area,Year,Unit,Value,Flag,Flag Description,Note
0,32,Argentina,1961,t,16000.0,A,Official figure,NaN
1,32,Argentina,1962,t,14000.0,A,Official figure,NaN
2,32,Argentina,1963,t,15000.0,A,Official figure,NaN
3,32,Argentina,1964,t,17000.0,A,Official figure,NaN
4,32,Argentina,1965,t,20000.0,A,Official figure,NaN


In [3]:
print(wb.shape)
print(wb.dtypes)
wb.head()

(266, 71)
Country Name       object
Country Code       object
Indicator Name     object
Indicator Code     object
1960              float64
                   ...   
2022              float64
2023              float64
2024              float64
2025              float64
Unnamed: 70       float64
Length: 71, dtype: object


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,28440.041688,30082.158423,30645.890602,22759.807175,26749.329609,30975.998912,35718.753119,39498.594129,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089204,186.909053,197.367547,225.400079,208.962717,226.836135,...,1528.104224,1552.073722,1507.085600,1351.591669,1562.416175,1679.327622,1571.449189,1615.396356,NaN,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,NaN,NaN,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,121.936832,127.451040,133.823783,139.004980,148.545883,155.561897,...,1574.230564,1720.140092,2216.385055,2030.861659,2112.794076,2138.473153,1841.855064,1411.337029,NaN,NaN
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2790.718869,2860.093648,2493.678844,1759.356199,2303.908127,3682.113151,2916.136633,2665.874448,NaN,NaN


## 2. Country Name Mapping

In [4]:
print('Countries:', wb['Country Name'].nunique())
print('Countries:', fao['Area'].nunique())

Countries: 266
Countries: 71


In [5]:
print(wb['Country Name'].unique())

['Aruba' 'Africa Eastern and Southern' 'Afghanistan'
 'Africa Western and Central' 'Angola' 'Albania' 'Andorra' 'Arab World'
 'United Arab Emirates' 'Argentina' 'Armenia' 'American Samoa'
 'Antigua and Barbuda' 'Australia' 'Austria' 'Azerbaijan' 'Burundi'
 'Belgium' 'Benin' 'Burkina Faso' 'Bangladesh' 'Bulgaria' 'Bahrain'
 'Bahamas, The' 'Bosnia and Herzegovina' 'Belarus' 'Belize' 'Bermuda'
 'Bolivia' 'Brazil' 'Barbados' 'Brunei Darussalam' 'Bhutan' 'Botswana'
 'Central African Republic' 'Canada' 'Central Europe and the Baltics'
 'Switzerland' 'Channel Islands' 'Chile' 'China' "Cote d'Ivoire"
 'Cameroon' 'Congo, Dem. Rep.' 'Congo, Rep.' 'Colombia' 'Comoros'
 'Cabo Verde' 'Costa Rica' 'Caribbean small states' 'Cuba' 'Curacao'
 'Cayman Islands' 'Cyprus' 'Czechia' 'Germany' 'Djibouti' 'Dominica'
 'Denmark' 'Dominican Republic' 'Algeria'
 'East Asia & Pacific (excluding high income)'
 'Early-demographic dividend' 'East Asia & Pacific'
 'Europe & Central Asia (excluding high income)' 'Europ

In [6]:
print(fao['Area'].unique())

['Argentina' 'Armenia' 'Austria' 'Belarus' 'Belgium' 'Bhutan'
 'Bolivia (Plurinational State of)' 'Botswana' 'Bulgaria' 'Burundi' 'Chad'
 'China' 'China, mainland' 'China, Taiwan Province of' 'Colombia' 'Congo'
 'Croatia' 'Czechia' 'Czechoslovakia' 'Eritrea' 'Estonia' 'Ethiopia'
 'Ethiopia PDR' 'Finland' 'France' 'Gambia' 'Georgia' 'Greece'
 'Guinea-Bissau' 'Hungary' 'Ireland' 'Italy' 'Jordan' 'Kazakhstan'
 'Latvia' 'Lithuania' 'Luxembourg' 'Mali' 'Malta' 'Mexico' 'Mongolia'
 'Morocco' 'Myanmar' 'New Zealand' 'Niger' 'Oman' 'Palestine' 'Peru'
 'Poland' 'Portugal' 'Qatar' 'Republic of Korea' 'Romania'
 'Russian Federation' 'Saudi Arabia' 'Senegal' 'Serbia' 'Sierra Leone'
 'Slovakia' 'Slovenia' 'South Africa' 'Spain' 'Switzerland' 'Thailand'
 'Togo' 'Tunisia' 'Ukraine' 'United Republic of Tanzania' 'USSR'
 'Uzbekistan' 'Zimbabwe']


In [7]:
for name in fao['Area'].unique():
    if name not in wb['Country Name'].values:
        print(name)

Bolivia (Plurinational State of)
China, mainland
China, Taiwan Province of
Congo
Czechoslovakia
Ethiopia PDR
Gambia
Palestine
Republic of Korea
Slovakia
United Republic of Tanzania
USSR


In [8]:
fao[fao['Area'] == 'China']

,Area Code (M49),Area,Year,Unit,Value,Flag,Flag Description,Note
525,159,China,1961,t,1000.00,E,Estimated value,NaN
526,159,China,1962,t,1000.00,E,Estimated value,NaN
527,159,China,1963,t,2000.00,E,Estimated value,NaN
528,159,China,1964,t,2000.00,E,Estimated value,NaN
529,159,China,1965,t,3000.00,E,Estimated value,NaN
...,...,...,...,...,...,...,...,...
584,159,China,2020,t,44931.04,E,Estimated value,NaN
585,159,China,2021,t,44091.55,E,Estimated value,NaN
586,159,China,2022,t,44385.73,E,Estimated value,NaN
587,159,China,2023,t,44470.34,E,Estimated value,NaN


In [9]:
fao[fao['Area'] == 'China, mainland'].head()

,Area Code (M49),Area,Year,Unit,Value,Flag,Flag Description,Note
597,156,"China, mainland",1999,t,20000.0,E,Estimated value,NaN
598,156,"China, mainland",2000,t,25000.0,E,Estimated value,NaN
599,156,"China, mainland",2001,t,30000.0,E,Estimated value,NaN
600,156,"China, mainland",2002,t,35000.0,E,Estimated value,NaN
601,156,"China, mainland",2003,t,30000.0,E,Estimated value,NaN


In [10]:
fao[fao['Area'] == 'China, Taiwan Province of']

,Area Code (M49),Area,Year,Unit,Value,Flag,Flag Description,Note
623,158,"China, Taiwan Province of",1961,t,1000.00,E,Estimated value,NaN
624,158,"China, Taiwan Province of",1962,t,1000.00,E,Estimated value,NaN
625,158,"China, Taiwan Province of",1963,t,2000.00,E,Estimated value,NaN
626,158,"China, Taiwan Province of",1964,t,2000.00,E,Estimated value,NaN
627,158,"China, Taiwan Province of",1965,t,3000.00,E,Estimated value,NaN
...,...,...,...,...,...,...,...,...
682,158,"China, Taiwan Province of",2020,t,402.64,I,Value imputed by a receiving agency,NaN
683,158,"China, Taiwan Province of",2021,t,402.07,I,Value imputed by a receiving agency,NaN
684,158,"China, Taiwan Province of",2022,t,402.18,I,Value imputed by a receiving agency,NaN
685,158,"China, Taiwan Province of",2023,t,403.19,I,Value imputed by a receiving agency,NaN


In [11]:
wb[wb['Country Name'] == 'China']

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
40,China,CHN,GDP per capita (current US$),NY.GDP.PCAP.CD,89.715075,75.965501,71.061685,74.468154,85.661107,98.668169,...,8979.676527,10085.663815,10342.900952,10627.463799,12887.435724,12970.605641,12951.17824,13303.148154,NaN,NaN


In [12]:
country_name_map = {
    'Bolivia (Plurinational State of)': 'Bolivia',
    'China, mainland':                  'China',
    'Congo':                            'Congo, Rep.',
    'Gambia':                           'Gambia, The',
    'Palestine':                        'West Bank and Gaza',
    'Republic of Korea':                'Korea, Rep.',
    'Slovakia':                         'Slovak Republic',
    'United Republic of Tanzania':      'Tanzania',
}

## 3. Clean FAO Dataset

In [13]:
drop_countries = ['China', 'China, Taiwan Province of', 'Czechoslovakia', 'Ethiopia PDR', 'USSR']
fao_clean = fao[~fao['Area'].isin(drop_countries)].copy()

fao_clean['Area'] = fao_clean['Area'].replace(country_name_map)

fao_clean = fao_clean[['Area', 'Year', 'Value', 'Flag', 'Flag Description']].copy()
fao_clean.columns = ['Country', 'Year', 'Cereal_Production_Value', 'Flag', 'Flag Description']

fao_clean

,Country,Year,Cereal_Production_Value,Flag,Flag Description
0,Argentina,1961,16000.00,A,Official figure
1,Argentina,1962,14000.00,A,Official figure
2,Argentina,1963,15000.00,A,Official figure
3,Argentina,1964,17000.00,A,Official figure
4,Argentina,1965,20000.00,A,Official figure
...,...,...,...,...,...
3003,Zimbabwe,2020,2290.28,I,Value imputed by a receiving agency
3004,Zimbabwe,2021,2306.29,I,Value imputed by a receiving agency
3005,Zimbabwe,2022,2293.19,I,Value imputed by a receiving agency
3006,Zimbabwe,2023,2296.52,I,Value imputed by a receiving agency


## 4. Reshape World Bank Dataset (Wide -> Long)

In [14]:
year_cols = [c for c in wb.columns if c.isdigit()]

wb_long = wb.melt(
    id_vars=['Country Name', 'Country Code'],
    value_vars=year_cols,
    var_name='Year',
    value_name='GDP_per_capita_USD'
)

wb_long.columns = ['Country', 'Country_Code', 'Year', 'GDP_per_capita_USD']
wb_long['Year'] = wb_long['Year'].astype(int)

print(wb_long.shape)
wb_long

(17556, 4)


,Country,Country_Code,Year,GDP_per_capita_USD
0,Aruba,ABW,1960,NaN
1,Africa Eastern and Southern,AFE,1960,186.089204
2,Afghanistan,AFG,1960,NaN
3,Africa Western and Central,AFW,1960,121.936832
4,Angola,AGO,1960,NaN
...,...,...,...,...
17551,Kosovo,XKX,2025,NaN
17552,"Yemen, Rep.",YEM,2025,NaN
17553,South Africa,ZAF,2025,NaN
17554,Zambia,ZMB,2025,NaN


## 5. Merge Datasets

In [15]:
merged = pd.merge(
    fao_clean,
    wb_long[['Country', 'Year', 'GDP_per_capita_USD']],
    on=['Country', 'Year'],
    how='inner'
)

print('Shape:', merged.shape)
print('Num of Countries:', merged['Country'].nunique())
print('Year range:', merged['Year'].min(), '~', merged['Year'].max())
merged

Shape: (2393, 6)
Num of Countries: 66
Year range: 1961 ~ 2024


,Country,Year,Cereal_Production_Value,Flag,Flag Description,GDP_per_capita_USD
0,Argentina,1961,16000.00,A,Official figure,971.338043
1,Argentina,1962,14000.00,A,Official figure,870.217491
2,Argentina,1963,15000.00,A,Official figure,852.972425
3,Argentina,1964,17000.00,A,Official figure,1176.200862
4,Argentina,1965,20000.00,A,Official figure,1281.833380
...,...,...,...,...,...,...
2388,Zimbabwe,2020,2290.28,I,Value imputed by a receiving agency,2059.674454
2389,Zimbabwe,2021,2306.29,I,Value imputed by a receiving agency,2613.605421
2390,Zimbabwe,2022,2293.19,I,Value imputed by a receiving agency,2536.400502
2391,Zimbabwe,2023,2296.52,I,Value imputed by a receiving agency,2195.224921


## 6. Check Merged Dataset

In [16]:
print('Missing values:')
print(merged.isnull().sum())

print('\nDuplicate rows:', merged.duplicated().sum())

print('\nCountries in merged dataset:')
print((merged['Country'].unique()))

Missing values:
Country                     0
Year                        0
Cereal_Production_Value     0
Flag                        0
Flag Description            0
GDP_per_capita_USD         91
dtype: int64

Duplicate rows: 0

Countries in merged dataset:
['Argentina' 'Armenia' 'Austria' 'Belarus' 'Belgium' 'Bhutan' 'Bolivia'
 'Botswana' 'Bulgaria' 'Burundi' 'Chad' 'China' 'Colombia' 'Congo, Rep.'
 'Croatia' 'Czechia' 'Eritrea' 'Estonia' 'Ethiopia' 'Finland' 'France'
 'Gambia, The' 'Georgia' 'Greece' 'Guinea-Bissau' 'Hungary' 'Ireland'
 'Italy' 'Jordan' 'Kazakhstan' 'Latvia' 'Lithuania' 'Luxembourg' 'Mali'
 'Malta' 'Mexico' 'Mongolia' 'Morocco' 'Myanmar' 'New Zealand' 'Niger'
 'Oman' 'West Bank and Gaza' 'Peru' 'Poland' 'Portugal' 'Qatar'
 'Korea, Rep.' 'Romania' 'Russian Federation' 'Saudi Arabia' 'Senegal'
 'Serbia' 'Sierra Leone' 'Slovak Republic' 'Slovenia' 'South Africa'
 'Spain' 'Switzerland' 'Thailand' 'Togo' 'Tunisia' 'Ukraine' 'Tanzania'
 'Uzbekistan' 'Zimbabwe']


## 7. Save Merged Dataset

In [17]:
merged.to_csv('../data/processed/merged_cereal_gdp.csv', index=False)